# Convex Cost Structure vs RL Pricing Accuracy
This notebook explores how the convex cost parameters (`c` and `gamma`) relate to the percent difference between LSM and RL swing option prices.

deleted lines from the CSV file "Convex Costs Results 2.csv"
```
0.3,1.5,11,0.09925343649705881,0.46132950219726565,364.79952581887306
1,0,14,2.6726823974709237,2.655080487548828,-0.6585859187291394
0.1,2,14,1.0145340884554201,1.368483587890625,34.88788631775558
0.8,3,14,-3.0863438318376355,0.47569408935546875,115.41286762830427
```

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

In [3]:
data_path = "Convex Costs Results 2.csv"
df = pd.read_csv(data_path)
df.head()

,c,gamma,ID,LSM,Best RL,PctDiff
0,0.00,1.0,14,2.684180,2.660009,-0.900494
1,0.01,1.5,13,2.507931,2.491421,-0.658301
2,0.01,1.0,13,2.536462,2.526979,-0.373853
3,0.01,2.0,12,2.456533,2.451500,-0.204877
4,0.01,3.0,12,2.218985,2.249093,1.356832


## Summary Statistics
Quick descriptive statistics for the main variables.

In [4]:
df.describe()

,c,gamma,ID,LSM,Best RL,PctDiff
count,26.000000,26.000000,26.000000,26.000000,26.000000,26.000000
mean,0.056538,1.711538,12.653846,1.796745,1.908035,12.263765
std,0.045602,0.680780,1.129329,0.591501,0.458609,25.350394
min,0.000000,1.000000,11.000000,0.484353,0.999328,-0.961274
25%,0.020000,1.000000,12.000000,1.320116,1.640111,-0.558747
50%,0.045000,1.500000,13.000000,1.861445,1.934455,0.950054
75%,0.080000,2.000000,14.000000,2.218672,2.239626,8.540146
max,0.150000,3.000000,14.000000,2.684180,2.660009,106.322350


## Pairwise Spearman Correlations
Spearman rank correlations capture monotonic relationships without assuming linearity.

In [5]:
spearman_corr = df[['c', 'gamma', 'PctDiff']].corr(method='spearman')
spearman_corr

,c,gamma,PctDiff
c,1.000000,-0.079429,0.431545
gamma,-0.079429,1.000000,0.737557
PctDiff,0.431545,0.737557,1.000000


## Scatter Visualization
Visualizing how `PctDiff` varies with `c` and `gamma`.

In [18]:
from itertools import cycle
from plotly.subplots import make_subplots
from plotly.colors import qualitative
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Use a renderer that loads MathJax (ensures LaTeX is rendered).
# Alternatives: "notebook_connected", "iframe_connected", "jupyterlab" (depending on your environment).
pio.renderers.default = "jupyterlab"

# --- Layout setup (tighter & balanced) ---
fig_plotly = make_subplots(
    rows=2, cols=2,
    subplot_titles=['PctDiff vs c', 'PctDiff vs gamma', '', ''],
    shared_yaxes=True,
    vertical_spacing=0.10,        # increased value for more space
    row_heights=[0.84, 0.16]
)

palette = qualitative.Plotly + qualitative.Safe

for idx, x_col in enumerate(['c', 'gamma'], start=1):
    # Color by the latent variable (gamma for left, c for right)
    color_var = 'gamma' if x_col == 'c' else 'c'
    unique_values = sorted(df[color_var].unique())
    palette_iter = cycle(palette)
    color_map = {val: next(palette_iter) for val in unique_values}

    # Scatter points by category with fixed colors
    for val in unique_values:
        mask = df[color_var] == val
        fig_plotly.add_trace(
            go.Scatter(
                x=df.loc[mask, x_col],
                y=df.loc[mask, 'PctDiff'],
                mode='markers',
                marker=dict(size=8, color=color_map[val]),
                hoverinfo='skip',
                showlegend=False
            ),
            row=1, col=idx
        )

    # Simple linear trend + ±1 SE band
    x_range = np.linspace(df[x_col].min(), df[x_col].max(), 200)
    pred_frame = pd.DataFrame({x_col: x_range})
    trend_fit = smf.ols(f'PctDiff ~ {x_col}', data=df).fit()
    pred_summary = trend_fit.get_prediction(pred_frame).summary_frame()
    mean = pred_summary['mean']
    se = pred_summary['mean_se']
    upper = mean + se
    lower = mean - se

    # ±1 SE shaded band
    fig_plotly.add_trace(
        go.Scatter(
            x=np.concatenate([x_range, x_range[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill='toself',
            fillcolor='rgba(31, 119, 180, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            mode='lines',
            showlegend=False
        ),
        row=1, col=idx
    )

    # Mean line
    fig_plotly.add_trace(
        go.Scatter(
            x=x_range, y=mean, mode='lines',
            line=dict(color='rgba(31, 119, 180, 1)', width=2),
            name='Trend', showlegend=False
        ),
        row=1, col=idx
    )

    # Axis titles for top row
    fig_plotly.update_xaxes(title_text=x_col, row=1, col=idx)

    # Legend panels (compact horizontal layout)
    legend_x = list(range(len(unique_values)))
    legend_y = [0] * len(unique_values)
    legend_colors = [color_map[v] for v in unique_values]
    legend_labels = [format(v, '.3g') for v in unique_values]

    fig_plotly.add_trace(
        go.Scatter(
            x=legend_x, y=legend_y,
            mode='markers+text',
            marker=dict(color=legend_colors, size=12),
            text=legend_labels, textposition='bottom center',
            hoverinfo='skip', showlegend=False
        ),
        row=2, col=idx
    )

    # Title for legend panel (lowered so it doesn't push layout)
    fig_plotly.add_annotation(
        x=0.5, y=1.02, xref=f'x{idx+2} domain', yref='y domain',
        text=f"<b>{color_var}</b>", showarrow=False, font=dict(size=12),
        row=2, col=idx
    )

    # Hide axes in legend panels
    fig_plotly.update_xaxes(visible=False, row=2, col=idx)
    fig_plotly.update_yaxes(visible=False, row=2, col=idx)

# --- Shared y-axis label (proper LaTeX) ---
y_axis_title = r"$\displaystyle PctDiff = \frac{\mathrm{RL} - \mathrm{LSM}}{\lvert \mathrm{LSM} \rvert}$"
fig_plotly.update_yaxes(title_text=y_axis_title, row=1, col=1, title_standoff=14, automargin=True)

# --- Global layout ---
fig_plotly.update_layout(
    title=dict(
        text='RL vs LSM Pricing Gap under Convex Costs',
        x=0.5, xanchor='center',
        y=0.98, yanchor='top',
        pad=dict(t=2, b=0)
    ),
    template='plotly_white',
    height=600, width=980,
    margin=dict(l=80, r=30, t=70, b=50),
    font=dict(size=12)
)

# Improve axis-label spacing
fig_plotly.update_xaxes(title_standoff=10, row=1, col=1)
fig_plotly.update_xaxes(title_standoff=10, row=1, col=2)
fig_plotly.update_yaxes(title_standoff=14, row=1, col=1)

# Show with renderer that loads MathJax
fig_plotly.show(renderer="jupyterlab")

## Robust Linear Model
Fit an ordinary least squares model with heteroskedasticity-robust standard errors to test whether `c`, `gamma`, and their interaction explain `PctDiff`.

In [7]:
model = smf.ols('PctDiff ~ c + gamma + c:gamma', data=df).fit()
robust_res = model.get_robustcov_results(cov_type='HC3')
robust_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                PctDiff   R-squared:                       0.808
Model:                            OLS   Adj. R-squared:                  0.782
Method:                 Least Squares   F-statistic:                     9.218
Date:                Sat, 08 Nov 2025   Prob (F-statistic):           0.000385
Time:                        15:13:55   Log-Likelihood:                -98.948
No. Observations:                  26   AIC:                             205.9
Df Residuals:                      22   BIC:                             210.9
Df Model:                           3                                         
Covariance Type:                  HC3                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.1756      9.954     -0.018      0.986     -20.819      20.467
c           -671.8654    322.586     -2.083      0.049   -1340.868      -2.863
gamma         -4.2463      7.286     -0.583      0.566     -19.357      10.864
c:gamma      628.9423    242.701      2.591      0.017     125.611    1132.274
==============================================================================
Omnibus:                        0.775   Durbin-Watson:                   2.134
Prob(Omnibus):                  0.679   Jarque-Bera (JB):                0.458
Skew:                           0.321   Prob(JB):                        0.795
Kurtosis:                       2.892   Cond. No.                         176.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
"""

## Effect Grid
Predicted percent difference across the observed grid of `c` and `gamma`.

In [8]:
grid = df[['c', 'gamma']].drop_duplicates().sort_values(['c', 'gamma']).reset_index(drop=True)
grid['pred_pct_diff'] = robust_res.predict(grid)
grid

,c,gamma,pred_pct_diff
0,0.00,1.0,-4.421937
1,0.01,1.0,-4.851168
2,0.01,1.5,-3.829617
3,0.01,2.0,-2.808066
4,0.01,3.0,-0.764965
5,0.02,1.0,-5.280399
6,0.02,1.5,-1.114137
7,0.02,2.0,3.052125
8,0.02,3.0,11.384649
9,0.04,1.0,-6.138862


## Interpretation
- The regression provides coefficient tests for `c`, `gamma`, and their interaction under robust standard errors.
- Combine the coefficient p-values with the Spearman correlations to assess whether higher convex costs correspond to larger RL advantages.
- Inspect the predicted grid to pinpoint regimes where RL diverges most from LSM.

In [21]:
print("Robust Regression Coefficients (PctDiff ~ c + gamma + c:gamma):\n")
for name, value in zip(robust_res.model.exog_names, robust_res.params):
    if name == "Intercept":
        desc = "Baseline PctDiff when c and gamma are zero"
    elif name == "c":
        desc = "Effect of c (convex cost parameter) on PctDiff"
    elif name == "gamma":
        desc = "Effect of gamma (convexity exponent) on PctDiff"
    elif name == "c:gamma":
        desc = "Interaction effect: how c's effect changes with gamma"
    else:
        desc = ""
    print(f"{name:10}: {value:10.4f}   # {desc}")

Robust Regression Coefficients (PctDiff ~ c + gamma + c:gamma):

Intercept :    -0.1756   # Baseline PctDiff when c and gamma are zero
c         :  -671.8654   # Effect of c (convex cost parameter) on PctDiff
gamma     :    -4.2463   # Effect of gamma (convexity exponent) on PctDiff
c:gamma   :   628.9423   # Interaction effect: how c's effect changes with gamma


In [19]:
robust_res.pvalues

array([0.98608268, 0.04910897, 0.56595244, 0.01665872])

In [49]:
pctdiff_pivot = (
    df.pivot_table(index='c', columns='gamma', values='PctDiff', aggfunc='mean')
      .sort_index()
      .reindex(sorted(df['gamma'].unique()), axis=1)
)

abs_max = np.nanmax(np.abs(pctdiff_pivot.values))
styled_pivot = (
    pctdiff_pivot.round(3)
    .style.format(precision=3)
    .background_gradient(cmap='coolwarm', axis=None, vmin=-abs_max, vmax=abs_max)
)
ctdiff_pivot = (
    df.pivot_table(index='c', columns='gamma', values='PctDiff', aggfunc='mean')
      .sort_index()
      .reindex(sorted(df['gamma'].unique()), axis=1)
)

def border_color(val):
    if pd.isna(val):
        return ''
    color = 'green' if val > 0 else 'red'
    return f'border: 2px solid {color};'

styled_pivot = (
    pctdiff_pivot.round(3)
    .style.format(precision=3)
    .background_gradient(cmap='coolwarm', axis=None, vmin=-abs_max, vmax=abs_max)
    .applymap(border_color)
)
styled_pivot

/var/folders/57/8q11hb450rz9z_rwvfbzpkmm0000gn/T/ipykernel_88912/110698141.py:29: FutureWarning:

Styler.applymap has been deprecated. Use Styler.map instead.



gamma,1.000000,1.500000,2.000000,3.000000
c,,,,
0.000000,-0.900,nan,nan,nan
0.010000,-0.374,-0.658,-0.205,1.357
0.020000,-0.239,-0.620,-0.294,9.166
0.040000,-0.961,0.382,4.778,42.560
0.050000,-0.874,0.543,6.661,67.940
0.080000,-0.782,3.541,20.844,nan
0.100000,-0.947,6.087,35.703,nan
0.150000,1.827,17.999,106.322,nan
